<a href="https://colab.research.google.com/github/OJB-Quantum/PDF-Table-to-SVG/blob/main/PDF_Table_to_SVG_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)

In [1]:
# 1. Update system package lists
!apt-get update

# 2. Install system-level binary dependencies required for PDF rasterization
!apt-get install -y poppler-utils

# 3. Bootstrap uv into the current Colab environment
!pip install uv -q

# 4. Use uv to rapidly install Python libraries into the system environment
!uv pip install --system -q pdf2image python-docx opencv-python-headless cupy-cuda12x

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,482 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,959 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,310 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,219 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main 

In [2]:
"""
Extract tables visually from documents using GPU-accelerated computer vision.

This module prompts the user to upload a PDF or DOCX file (preferrably PDF), rasterizes the
pages, and utilizes CuPy (CUDA-accelerated NumPy) to perform morphological
operations to detect table bounding boxes. The extracted high-resolution
tables are then embedded within an SVG container and served to the user
as interactive Base64 download links.
"""

import base64
import io
import os
import zipfile
from typing import List, Tuple

import cv2
import cupy as cp
import numpy as np
from cupyx.scipy import ndimage
from google.colab import files
from IPython.display import HTML, SVG, display
from pdf2image import convert_from_path


# =====================================================================
# CONTROL KNOBS: Parameterize your extraction and rendering here.
# =====================================================================
SCAN_DPI = 1200                 # Resolution for rasterizing PDF pages.
MORPH_KERNEL_LENGTH = 50       # Length of the kernel for line detection.
BINARIZATION_THRESH = 200      # Pixel intensity threshold (0-255).
PADDING_PX = 2                # Padding added around detected tables.
TARGET_WIDTH_PX = 3000         # The default width specified in the SVG.
# =====================================================================


def extract_images_from_docx(docx_path: str) -> List[np.ndarray]:
    """
    Extract embedded images from a DOCX file by unzipping its archive.

    Args:
        docx_path: File system path to the .docx document.

    Returns:
        A list of images loaded as NumPy arrays.
    """
    images = []
    with zipfile.ZipFile(docx_path, 'r') as docx_zip:
        for item in docx_zip.namelist():
            if item.startswith('word/media/'):
                img_data = docx_zip.read(item)
                img_array = np.frombuffer(img_data, np.uint8)
                img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
                if img is not None:
                    images.append(img)
    return images


def get_document_images(file_path: str) -> List[np.ndarray]:
    """
    Route the file to the appropriate rasterization or extraction engine.

    Args:
        file_path: File system path to the uploaded document.

    Returns:
        A list of images representing pages or embedded images.
    """
    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.pdf':
        print(f"Rasterizing PDF at {SCAN_DPI} DPI...")
        pil_images = convert_from_path(file_path, dpi=SCAN_DPI)
        # Convert PIL images to OpenCV format (BGR)
        return [cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR) for img in pil_images]
    elif ext == '.docx':
        print("Extracting embedded images from DOCX...")
        return extract_images_from_docx(file_path)
    else:
        print(f"Unsupported file extension: {ext}")
        return []


def detect_tables_gpu(img_array: np.ndarray) -> List[Tuple[int, int, int, int]]:
    """
    Utilize CuPy to detect tabular structures via morphological operations.

    Args:
        img_array: The high-resolution page image as a NumPy array.

    Returns:
        A list of bounding boxes (x, y, width, height) for detected tables.
    """
    # 1. Transfer matrix from CPU RAM to GPU VRAM
    img_gpu = cp.asarray(img_array)

    # 2. Convert to grayscale using standard luminosity weights
    weights = cp.array([0.1140, 0.5870, 0.2989]) # BGR order
    gray_gpu = cp.dot(img_gpu[..., :3], weights).astype(cp.uint8)

    # 3. Invert and binarize (text and lines become active/white)
    thresh_gpu = gray_gpu < BINARIZATION_THRESH

    # 4. Perform mathematical morphology to isolate lines
    # Using the 'size' parameter satisfies CuPy's internal filter requirements
    h_lines = ndimage.grey_opening(thresh_gpu, size=(1, MORPH_KERNEL_LENGTH))
    v_lines = ndimage.grey_opening(thresh_gpu, size=(MORPH_KERNEL_LENGTH, 1))

    # 5. Combine lines to find the grid and transfer back to CPU
    table_mask_gpu = h_lines | v_lines
    table_mask_cpu = cp.asnumpy(table_mask_gpu).astype(np.uint8) * 255

    # 6. Extract bounding boxes using CPU (OpenCV handles contours efficiently)
    contours, _ = cv2.findContours(
        table_mask_cpu,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    bounding_boxes = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        # Filter out noise by enforcing a minimum area
        if w > 100 and h > 100:
            bounding_boxes.append((x, y, w, h))

    # Sort boxes top-to-bottom
    bounding_boxes.sort(key=lambda b: b[1])
    return bounding_boxes


def generate_svg_with_embedded_raster(
    img_crop: np.ndarray,
    table_index: int
) -> None:
    """
    Embed a high-resolution raster crop into an SVG for absolute fidelity.

    Args:
        img_crop: The cropped table image array.
        table_index: An integer identifier for the current table.
    """
    # Encode the cropped NumPy array to a PNG byte stream
    _, buffer = cv2.imencode('.png', img_crop)
    b64_encoded = base64.b64encode(buffer).decode('utf-8')
    image_uri = f"data:image/png;base64,{b64_encoded}"

    orig_height, orig_width = img_crop.shape[:2]

    # Calculate aspect ratio to maintain proportions at TARGET_WIDTH_PX
    aspect_ratio = orig_height / orig_width
    target_height = int(TARGET_WIDTH_PX * aspect_ratio)

    # Construct the SVG DOM string
    svg_content = f"""<svg xmlns="http://www.w3.org/2000/svg"
         width="{TARGET_WIDTH_PX}" height="{target_height}"
         viewBox="0 0 {orig_width} {orig_height}">
        <image href="{image_uri}" x="0" y="0"
               width="{orig_width}" height="{orig_height}" />
    </svg>"""

    filename = f"extracted_table_{table_index + 1}.svg"

    # Generate HTML download button using the SVG content as a data URI
    svg_b64 = base64.b64encode(svg_content.encode('utf-8')).decode('utf-8')
    svg_uri = f"data:image/svg+xml;base64,{svg_b64}"

    html_button = f"""
    <a href="{svg_uri}" download="{filename}"
       style="display: inline-block; padding: 8px 16px; margin-bottom: 20px;
              background-color: #28a745; color: white; text-decoration: none;
              border-radius: 4px; font-family: sans-serif; font-size: 14px;">
       💾 Download {filename}
    </a>
    <br>
    """

    print(f"\n--- Processed Table {table_index + 1} ---")
    display(HTML(html_button))
    display(SVG(svg_content))


def main() -> None:
    """
    Execute the main interactive sequence for GPU visual table extraction.
    """
    print("Please upload your target .pdf or .docx file:")
    uploaded_files = files.upload()

    if not uploaded_files:
        print("No file was uploaded. Terminating process.")
        return

    table_counter = 0

    for file_name in uploaded_files.keys():
        print(f"\nProcessing document: {file_name}")

        pages = get_document_images(file_name)
        if not pages:
            continue

        print(f"Rasterized {len(pages)} page(s)/image(s). Initiating GPU scan...")

        for page_idx, page_img in enumerate(pages):
            boxes = detect_tables_gpu(page_img)

            for (x, y, w, h) in boxes:
                # Apply padding, ensuring we stay within image boundaries
                img_h, img_w = page_img.shape[:2]
                x1 = max(0, x - PADDING_PX)
                y1 = max(0, y - PADDING_PX)
                x2 = min(img_w, x + w + PADDING_PX)
                y2 = min(img_h, y + h + PADDING_PX)

                table_crop = page_img[y1:y2, x1:x2]
                # Convert BGR back to RGB for correct browser rendering
                table_crop_rgb = cv2.cvtColor(table_crop, cv2.COLOR_BGR2RGB)

                generate_svg_with_embedded_raster(table_crop_rgb, table_counter)
                table_counter += 1

    if table_counter == 0:
        print("No tables detected using the current morphological parameters.")


if __name__ == "__main__":
    main()

Please upload your target .pdf or .docx file:


No file was uploaded. Terminating process.
